# Interactive Protein Inference Notebook

This notebook provides an interactive interface for:
1. Selecting protein datasets
2. Choosing model types and configurations
3. Running inference with various options
4. Computing and visualizing metrics

## Setup

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Any
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# Lobster imports
from lobster.model import (
    LobsterPMLM,
    LobsterCBMPMLM,
    LobsterPCLM,
    LobsterConditionalPMLM,
    UME
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Dataset Selection

Choose your protein dataset source:

In [ ]:
# Dataset source selection
dataset_source = widgets.RadioButtons(
    options=['FASTA File', 'Direct Input', 'Example Sequences'],
    description='Data Source:',
    disabled=False
)

# FASTA file path
fasta_path = widgets.Text(
    value='test_data/query.fasta',
    description='FASTA Path:',
    disabled=False
)

# Direct sequence input
direct_sequence = widgets.Textarea(
    value='MGAGASAEEKHSRELEKKLKEDAEKDARTVKLLLLGAGESGKSTIVKQMKIIHQDGYSLEECLEFIAIIYGNTLQSILAIVRAMTTLNIQYGDSARQDDARKLMHMADTIEEGTMPKEMSDIIQRLWKDSGIQACFERASEYQLNDSAGYYLSDLERLVTPGYVPTEQDVLRSRVKTTGIIETQFSFKDLNFRMFDVGGQRSERKKWIHCFEGVTCIIFIAALSAYDMVLVEDDEVNRMHESLHLFNSICNHRYFATTSIVLFLNKKDVFFEKIKKAHLSICFPDYDGPNTYEDAGNYIKVQFLELNMRRDVKEIYSHMTCATDTQNVKFVFDAVTDIIIKENLKDCGLF',
    description='Sequence:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='100px')
)

# Example sequences dropdown
example_sequences = {
    'G-protein alpha subunit': 'MGAGASAEEKHSRELEKKLKEDAEKDARTVKLLLLGAGESGKSTIVKQMKIIHQDGYSLEECLEFIAIIYGNTLQSILAIVRAMTTLNIQYGDSARQDDARKLMHMADTIEEGTMPKEMSDIIQRLWKDSGIQACFERASEYQLNDSAGYYLSDLERLVTPGYVPTEQDVLRSRVKTTGIIETQFSFKDLNFRMFDVGGQRSERKKWIHCFEGVTCIIFIAALSAYDMVLVEDDEVNRMHESLHLFNSICNHRYFATTSIVLFLNKKDVFFEKIKKAHLSICFPDYDGPNTYEDAGNYIKVQFLELNMRRDVKEIYSHMTCATDTQNVKFVFDAVTDIIIKENLKDCGLF',
    'Small peptide': 'MKFLKFSLLTAVLLSVVFAFSSCGDDDDTISSSTTGPPSPDLSRIVGGWECELGDNMECFTFKYGGCMGIGNRNNNFKTEECL',
    'Antibody VH': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFTDYYMHWVRQAPGQGLEWMGWINPNSGGTNYAQKFQGRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR'
}

example_dropdown = widgets.Dropdown(
    options=list(example_sequences.keys()),
    description='Example:',
    disabled=False
)

display(dataset_source)
display(HTML('<b>Option 1: FASTA File</b>'))
display(fasta_path)
display(HTML('<b>Option 2: Direct Sequence Input</b>'))
display(direct_sequence)
display(HTML('<b>Option 3: Example Sequences</b>'))
display(example_dropdown)

In [ ]:
# Load sequences based on selection
def load_sequences():
    """Load sequences based on user selection."""
    sequences = []
    seq_ids = []
    
    if dataset_source.value == 'FASTA File':
        if os.path.exists(fasta_path.value):
            for record in SeqIO.parse(fasta_path.value, 'fasta'):
                sequences.append(str(record.seq))
                seq_ids.append(record.id)
            print(f"✓ Loaded {len(sequences)} sequences from {fasta_path.value}")
        else:
            print(f"❌ File not found: {fasta_path.value}")
            return None, None
    
    elif dataset_source.value == 'Direct Input':
        seq = direct_sequence.value.strip().replace(' ', '').replace('\n', '')
        sequences = [seq]
        seq_ids = ['user_sequence_1']
        print(f"✓ Loaded 1 sequence from direct input (length: {len(seq)})")
    
    elif dataset_source.value == 'Example Sequences':
        seq = example_sequences[example_dropdown.value]
        sequences = [seq]
        seq_ids = [example_dropdown.value]
        print(f"✓ Loaded example: {example_dropdown.value} (length: {len(seq)})")
    
    return sequences, seq_ids

# Load the sequences
sequences, seq_ids = load_sequences()

# Display sequence information
if sequences:
    print(f"\nSequence Summary:")
    for i, (seq_id, seq) in enumerate(zip(seq_ids[:5], sequences[:5])):
        print(f"  {i+1}. {seq_id}: {len(seq)} amino acids")
        print(f"     {seq[:50]}..." if len(seq) > 50 else f"     {seq}")
    if len(sequences) > 5:
        print(f"  ... and {len(sequences) - 5} more sequences")

## 2. Model Selection

Choose the model type and configuration:

In [ ]:
# Model selection
model_type = widgets.Dropdown(
    options=[
        ('Lobster PMLM (Masked LM)', 'pmlm'),
        ('Lobster CBM-PMLM (Concept Bottleneck)', 'cbm'),
        ('Lobster PCLM (Causal LM)', 'pclm'),
        ('UME (Multimodal)', 'ume')
    ],
    value='pmlm',
    description='Model Type:',
)

# Model checkpoint/identifier
model_checkpoint = widgets.Text(
    value='asalam91/lobster_24M',
    description='Checkpoint:',
    layout=widgets.Layout(width='50%')
)

# Device selection
device_widget = widgets.RadioButtons(
    options=['auto', 'cuda', 'cpu'],
    value='auto',
    description='Device:'
)

# Update checkpoint when model type changes
def on_model_change(change):
    if change['new'] == 'pmlm':
        model_checkpoint.value = 'asalam91/lobster_24M'
    elif change['new'] == 'cbm':
        model_checkpoint.value = 'asalam91/cb_lobster_24M'
    elif change['new'] == 'pclm':
        model_checkpoint.value = 'asalam91/lobster_clm_24M'
    elif change['new'] == 'ume':
        model_checkpoint.value = 'ume-mini-base-12M'

model_type.observe(on_model_change, names='value')

display(model_type)
display(model_checkpoint)
display(device_widget)

print("\nModel descriptions:")
print("  • PMLM: Masked language model for protein embeddings (BERT-style)")
print("  • CBM-PMLM: Concept bottleneck model with 718 interpretable biological concepts")
print("  • PCLM: Causal language model for protein generation (GPT-style)")
print("  • UME: Multimodal embeddings for proteins")

In [ ]:
# Load the model
device = 'cuda' if (device_widget.value == 'auto' and torch.cuda.is_available()) else device_widget.value
device = 'cuda' if device == 'auto' else device

print(f"Loading model on device: {device}")
print(f"Model type: {model_type.value}")
print(f"Checkpoint: {model_checkpoint.value}")

try:
    if model_type.value == 'pmlm':
        model = LobsterPMLM(model_checkpoint.value).to(device)
    elif model_type.value == 'cbm':
        model = LobsterCBMPMLM(model_checkpoint.value).to(device)
    elif model_type.value == 'pclm':
        model = LobsterPCLM(model_checkpoint.value).to(device)
    elif model_type.value == 'ume':
        model = UME.from_pretrained(model_checkpoint.value).to(device)
    
    model.eval()
    print("✓ Model loaded successfully!")
    
    # Display model info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Statistics:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model = None

## 3. Run Inference

Generate embeddings and representations:

In [ ]:
# Inference options
pooling_method = widgets.Dropdown(
    options=['CLS token', 'Mean pooling', 'Max pooling', 'Both CLS and Mean'],
    value='Both CLS and Mean',
    description='Pooling:'
)

batch_size = widgets.IntSlider(
    value=8,
    min=1,
    max=64,
    step=1,
    description='Batch Size:'
)

display(pooling_method)
display(batch_size)

In [ ]:
# Run inference
if model is None or sequences is None:
    print("❌ Please load model and sequences first")
else:
    print("Running inference...")
    
    results = {}
    
    with torch.inference_mode():
        # Get latent representations
        latents = model.sequences_to_latents(sequences)[-1]
        print(f"✓ Generated latent representations: {latents.shape}")
        
        # Apply pooling
        if pooling_method.value in ['CLS token', 'Both CLS and Mean']:
            cls_embeddings = latents[:, 0, :]
            results['cls_embeddings'] = cls_embeddings
            print(f"✓ CLS token embeddings: {cls_embeddings.shape}")
        
        if pooling_method.value in ['Mean pooling', 'Both CLS and Mean']:
            mean_embeddings = torch.mean(latents, dim=1)
            results['mean_embeddings'] = mean_embeddings
            print(f"✓ Mean pooled embeddings: {mean_embeddings.shape}")
        
        if pooling_method.value == 'Max pooling':
            max_embeddings = torch.max(latents, dim=1)[0]
            results['max_embeddings'] = max_embeddings
            print(f"✓ Max pooled embeddings: {max_embeddings.shape}")
        
        # For CBM models, also get concepts
        if model_type.value == 'cbm':
            concepts = model.sequences_to_concepts(sequences)[-1]
            results['concepts'] = concepts
            print(f"✓ Extracted concepts: {concepts.shape}")
            print(f"\nAvailable concepts: {len(model.list_supported_concept())}")
    
    print("\n✓ Inference complete!")

## 4. Compute Metrics

Calculate biological and embedding metrics:

In [ ]:
# Compute sequence-level metrics
def compute_sequence_metrics(sequences: list[str]) -> pd.DataFrame:
    """Compute biological metrics for protein sequences."""
    metrics_data = []
    
    for seq_id, seq in zip(seq_ids, sequences):
        try:
            analyzer = ProteinAnalysis(seq)
            metrics = {
                'seq_id': seq_id,
                'length': len(seq),
                'molecular_weight': analyzer.molecular_weight(),
                'aromaticity': analyzer.aromaticity(),
                'instability_index': analyzer.instability_index(),
                'isoelectric_point': analyzer.isoelectric_point(),
                'gravy': analyzer.gravy(),
                'helix_fraction': analyzer.secondary_structure_fraction()[0],
                'turn_fraction': analyzer.secondary_structure_fraction()[1],
                'sheet_fraction': analyzer.secondary_structure_fraction()[2],
            }
            metrics_data.append(metrics)
        except Exception as e:
            print(f"Warning: Could not compute metrics for {seq_id}: {e}")
    
    return pd.DataFrame(metrics_data)

# Compute metrics
if sequences:
    metrics_df = compute_sequence_metrics(sequences)
    print("Sequence Metrics:")
    display(metrics_df)
    
    # Summary statistics
    print("\nSummary Statistics:")
    display(metrics_df.describe())

## 5. Visualize Results

Generate plots and visualizations:

In [ ]:
# Visualize embedding space (PCA/t-SNE)
if 'cls_embeddings' in results or 'mean_embeddings' in results:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    # Choose embedding
    emb_key = 'cls_embeddings' if 'cls_embeddings' in results else 'mean_embeddings'
    embeddings = results[emb_key].cpu().numpy()
    
    # PCA
    if len(embeddings) > 1:
        n_components = min(2, len(embeddings) - 1)
        pca = PCA(n_components=n_components)
        pca_embeddings = pca.fit_transform(embeddings)
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # PCA plot
        axes[0].scatter(pca_embeddings[:, 0], pca_embeddings[:, 1], alpha=0.6, s=100)
        for i, seq_id in enumerate(seq_ids[:len(pca_embeddings)]):
            axes[0].annotate(seq_id, (pca_embeddings[i, 0], pca_embeddings[i, 1]),
                           fontsize=8, alpha=0.7)
        axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} var)')
        axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} var)' if n_components > 1 else 'PC2')
        axes[0].set_title('PCA of Protein Embeddings')
        axes[0].grid(True, alpha=0.3)
        
        # Embedding norm distribution
        norms = np.linalg.norm(embeddings, axis=1)
        axes[1].hist(norms, bins=20, alpha=0.7, edgecolor='black')
        axes[1].set_xlabel('Embedding Norm')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Distribution of Embedding Norms')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("Need at least 2 sequences for PCA visualization")

In [ ]:
# Visualize concepts (for CBM models)
if 'concepts' in results and model_type.value == 'cbm':
    concepts_array = results['concepts'].cpu().numpy()
    concept_names = model.list_supported_concept()
    
    # Show top concepts for first sequence
    if len(sequences) > 0:
        seq_concepts = concepts_array[0]
        
        # Get top 10 positive and negative concepts
        top_indices = np.argsort(seq_concepts)[-10:][::-1]
        bottom_indices = np.argsort(seq_concepts)[:10]
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Top concepts
        top_concepts = [concept_names[i] for i in top_indices]
        top_values = [seq_concepts[i] for i in top_indices]
        axes[0].barh(range(len(top_concepts)), top_values, color='green', alpha=0.7)
        axes[0].set_yticks(range(len(top_concepts)))
        axes[0].set_yticklabels(top_concepts)
        axes[0].set_xlabel('Concept Value')
        axes[0].set_title(f'Top 10 Concepts for {seq_ids[0]}')
        axes[0].grid(True, alpha=0.3, axis='x')
        
        # Bottom concepts
        bottom_concepts = [concept_names[i] for i in bottom_indices]
        bottom_values = [seq_concepts[i] for i in bottom_indices]
        axes[1].barh(range(len(bottom_concepts)), bottom_values, color='red', alpha=0.7)
        axes[1].set_yticks(range(len(bottom_concepts)))
        axes[1].set_yticklabels(bottom_concepts)
        axes[1].set_xlabel('Concept Value')
        axes[1].set_title(f'Bottom 10 Concepts for {seq_ids[0]}')
        axes[1].grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.show()
        
        # Concept distribution heatmap (if multiple sequences)
        if len(sequences) > 1:
            plt.figure(figsize=(12, max(6, len(sequences) * 0.3)))
            sns.heatmap(concepts_array, cmap='RdYlGn', center=0,
                       yticklabels=seq_ids[:len(concepts_array)],
                       xticklabels=False,
                       cbar_kws={'label': 'Concept Value'})
            plt.xlabel('Concept Index')
            plt.ylabel('Sequence')
            plt.title('Concept Activation Heatmap')
            plt.tight_layout()
            plt.show()

In [ ]:
# Visualize sequence metrics
if not metrics_df.empty and len(metrics_df) > 1:
    # Select numeric columns
    numeric_cols = metrics_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Create subplots for different metrics
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Length distribution
    axes[0, 0].hist(metrics_df['length'], bins=20, alpha=0.7, edgecolor='black', color='skyblue')
    axes[0, 0].set_xlabel('Sequence Length')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Sequence Length Distribution')
    axes[0, 0].grid(True, alpha=0.3)
    
    # GRAVY vs Aromaticity
    axes[0, 1].scatter(metrics_df['gravy'], metrics_df['aromaticity'], alpha=0.6, s=100)
    axes[0, 1].set_xlabel('GRAVY (Hydrophobicity)')
    axes[0, 1].set_ylabel('Aromaticity')
    axes[0, 1].set_title('GRAVY vs Aromaticity')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Secondary structure composition
    ss_means = metrics_df[['helix_fraction', 'turn_fraction', 'sheet_fraction']].mean()
    axes[1, 0].bar(ss_means.index, ss_means.values, alpha=0.7,
                   color=['#ff9999', '#66b3ff', '#99ff99'], edgecolor='black')
    axes[1, 0].set_ylabel('Average Fraction')
    axes[1, 0].set_title('Average Secondary Structure Composition')
    axes[1, 0].set_xticklabels(['Helix', 'Turn', 'Sheet'])
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Molecular weight vs pI
    axes[1, 1].scatter(metrics_df['molecular_weight'], metrics_df['isoelectric_point'],
                      alpha=0.6, s=100, c=metrics_df['length'], cmap='viridis')
    axes[1, 1].set_xlabel('Molecular Weight (Da)')
    axes[1, 1].set_ylabel('Isoelectric Point')
    axes[1, 1].set_title('Molecular Weight vs Isoelectric Point')
    axes[1, 1].grid(True, alpha=0.3)
    cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
    cbar.set_label('Sequence Length')
    
    plt.tight_layout()
    plt.show()

## 6. Export Results

Save embeddings and metrics to files:

In [ ]:
# Export options
output_dir = widgets.Text(
    value='outputs/inference_results',
    description='Output Dir:',
    layout=widgets.Layout(width='50%')
)

export_button = widgets.Button(
    description='Export Results',
    button_style='success',
    icon='save'
)

output_status = widgets.Output()

def export_results(b):
    with output_status:
        clear_output()
        
        # Create output directory
        os.makedirs(output_dir.value, exist_ok=True)
        print(f"Exporting to: {output_dir.value}")
        
        # Save embeddings
        for key, value in results.items():
            if isinstance(value, torch.Tensor):
                np.save(f"{output_dir.value}/{key}.npy", value.cpu().numpy())
                print(f"✓ Saved {key}.npy")
        
        # Save metrics
        if not metrics_df.empty:
            metrics_df.to_csv(f"{output_dir.value}/sequence_metrics.csv", index=False)
            print(f"✓ Saved sequence_metrics.csv")
        
        # Save sequence IDs
        with open(f"{output_dir.value}/sequence_ids.txt", 'w') as f:
            f.write('\n'.join(seq_ids))
        print(f"✓ Saved sequence_ids.txt")
        
        print("\n✓ Export complete!")

export_button.on_click(export_results)

display(output_dir)
display(export_button)
display(output_status)